# 3장 컨텍스트

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `03_컨텍스트.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers
# 3장의 예제 3-1과 실습 3-1은 1장의 토크나이저만 씁니다(모델은 내려받지 않습니다)
import embed
LOCAL_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(LOCAL_MODEL)          # 토크나이저가 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
print('토크나이저:', LOCAL_MODEL)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 3-1. 채팅 템플릿 보기

교재의 절: "역할과 채팅 템플릿"

시스템 프롬프트, 질문, 이전 답, 새 질문으로 된 메시지 네 개를 1장의 토크나이저로 채팅 템플릿에 넣고, 만들어진 글과 그 토큰 수를 봅니다. 역할 이름 앞뒤에 어떤 표시 토큰이 붙는지, 마지막 줄이 무엇인지 봅니다.

In [ ]:
messages = [
    {'role': 'system', 'content': '너는 반도체 후공정 라인의 문의에 답하는 도우미다.'},
    {'role': 'user', 'content': '다이싱이 무엇인가요?'},
    {'role': 'assistant', 'content': '웨이퍼를 개별 칩으로 자르는 공정입니다.'},
    {'role': 'user', 'content': '그다음 공정은요?'},
]
# add_generation_prompt=True: 마지막에 모델의 답 차례를 여는 표시를 붙입니다
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(text)
print(f'메시지 {len(messages)}개 → 글 한 덩어리, {len(tokenizer.encode(text))}토큰')

<|im_start|>system
너는 반도체 후공정 라인의 문의에 답하는 도우미다.<|im_end|>
<|im_start|>user
다이싱이 무엇인가요?<|im_end|>
<|im_start|>assistant
웨이퍼를 개별 칩으로 자르는 공정입니다.<|im_end|>
<|im_start|>user
그다음 공정은요?<|im_end|>
<|im_start|>assistant

메시지 4개 → 글 한 덩어리, 78토큰


## 예제 3-2. 기억하지 않는 모델

교재의 절: "모델은 대화를 기억하지 않는다"

이름을 알려 준 뒤, 이력 없이 새 요청으로 이름을 물어봅니다. 이어서 이력을 함께 보내며 두 번 묻고, 요청마다 입력 토큰이 어떻게 늘어나는지 봅니다. `llm.ask()`는 공통 호출 함수 `llm.py`의 함수로, `history`에 이력을 주면 이력 뒤에 질문을 붙여 보냅니다(5장 "LLM API"). 이 셀은 API를 4번 호출합니다.

In [ ]:
intro = '제 이름은 김후공입니다. 기억해 주세요.'
first = llm.ask(intro, model='소형', max_tokens=500)
alone = llm.ask('제 이름이 뭐였죠?', model='소형', max_tokens=500)   # 이력 없이 새 요청
print(f'[이력 없이] 입력 토큰 {alone.input_tokens}개: {alone.text.strip()}\n')

history = [{'role': 'user', 'content': intro}, {'role': 'assistant', 'content': first.text}]
for question in ['제 이름이 뭐였죠?', '제가 처음에 한 말을 그대로 적어 주세요.']:
    reply = llm.ask(question, history=history, model='소형', max_tokens=500)   # 이력을 함께 보냄
    print(f'[이력 {len(history)}개와 함께] 입력 토큰 {reply.input_tokens}개: {reply.text.strip()}\n')
    history += [{'role': 'user', 'content': question}, {'role': 'assistant', 'content': reply.text}]

[이력 없이] 입력 토큰 21개: 죄송하지만 저는 대화 간에 기억을 유지하지 못해서, 이전 대화에서 이름을 알려드렸는지 알 수 없습니다. 제가 특정 이름을 가지고 있는 것도 아니고요. 다만 저는 Claude라는 AI 어시스턴트입니다.

혹시 제가 이름을 기억하길 원하셨던 거라면, 지금 알려주시면 이 대화 안에서는 그 이름을 기준으로 이야기할 수 있어요. 아니면 찾으시는 다른 정보가 있을까요?

[이력 2개와 함께] 입력 토큰 203개: 안내해 주신 대로 김후공 님이시죠. 이 대화 안에서 기억하고 있습니다.

[이력 4개와 함께] 입력 토큰 270개: 처음에 하신 말씀은 다음과 같습니다.

"제 이름은 김후공입니다. 기억해 주세요."


## 예제 3-3. 긴 문서의 중간

교재의 절: "긴 문서"

코드로 만든 가상 교대 일지 600줄의 앞(5%), 중간(50%), 끝(95%)에 부품 교체 기록 한 줄을 넣고, 매번 같은 질문을 보냅니다. 위치마다 기록을 찾는지, 입력 토큰이 얼마인지 봅니다. 이 셀은 API를 3번 호출합니다.

In [ ]:
equipment = ['1호기 다이 본더', '2호기 와이어 본더', '4호기 몰딩기', '5호기 테스트 핸들러', '6호기 마킹기']
lines = [f'{n // 3 + 1}일 {"ABC"[n % 3]}조: {equipment[n % 5]} 정상 가동, 생산 {400 + n % 37}로트, 특이 사항 없음.'
         for n in range(600)]
fact = '3호기 와이어 본더의 캐필러리를 교체함. 사유는 본딩 강도 저하.'
question = '캐필러리를 교체한 설비와 그 사유를 한 문장으로 답해 주세요.'
for position in [0.05, 0.5, 0.95]:                                # 일지의 앞, 중간, 끝
    log = lines.copy()
    log.insert(int(len(log) * position), fact)
    prompt = '다음은 교대 일지입니다.\n\n' + '\n'.join(log) + '\n\n' + question
    reply = llm.ask(prompt, model='소형', max_tokens=1000)
    print(f'위치 {position:.0%}: 입력 토큰 {reply.input_tokens:,}개, {reply.seconds:.1f}초 → {reply.text.strip()}')

위치 5%: 입력 토큰 25,656개, 1.9초 → 3호기 와이어 본더의 캐필러리를 교체했으며, 교체 사유는 본딩 강도 저하입니다.
위치 50%: 입력 토큰 25,656개, 1.2초 → 3호기 와이어 본더의 캐필러리를 교체했으며, 사유는 본딩 강도 저하입니다.
위치 95%: 입력 토큰 25,656개, 1.1초 → 3호기 와이어 본더의 캐필러리를 교체했으며, 사유는 본딩 강도 저하였습니다.


## 실습 3-1. 이력 줄이기

교재의 절: "개념 확인 실습"

시스템 메시지와 마지막 질문은 남기고, 가장 오래된 질문과 답부터 한 쌍씩 빼서 채팅 템플릿 기준 토큰 수를 `token_limit` 이하로 줄이는 `trim_history()`를 작성합니다. `token_limit=120`이면 몇 쌍이 남을지 먼저 `my_guess`에 적습니다. 결과를 보고, 오래된 대화를 뺄 때 모델이 무엇을 잃는지 한 문장으로 설명하면 끝납니다.

In [ ]:
def count(messages):
    """메시지 목록을 채팅 템플릿으로 바꾼 글의 토큰 수"""
    return len(tokenizer.encode(tokenizer.apply_chat_template(messages, tokenize=False)))


def trim_history(messages, token_limit):
    """시스템 메시지와 마지막 질문은 남기고, 가장 오래된 질문과 답부터 한 쌍씩 빼서 token_limit 이하로 줄입니다"""
    return messages  # TODO


history = [{'role': 'system', 'content': '너는 후공정 라인의 문의에 답하는 도우미다.'}]
for question, answer in [('다이싱이 무엇인가요?', '웨이퍼를 개별 칩으로 자르는 공정입니다.'),
                         ('다이 어태치는요?', '자른 칩을 기판이나 리드 프레임에 붙이는 공정입니다.'),
                         ('와이어 본딩은요?', '칩의 패드와 리드를 가는 금속선으로 잇는 공정입니다.'),
                         ('몰딩은요?', '칩과 와이어를 수지로 감싸 보호하는 공정입니다.')]:
    history += [{'role': 'user', 'content': question}, {'role': 'assistant', 'content': answer}]
history.append({'role': 'user', 'content': '마킹은 언제 하나요?'})

my_guess = None  # TODO: token_limit=120이면 질문과 답이 몇 쌍 남을지 먼저 적습니다
trimmed = trim_history(history, 120)
print(f'{count(history)}토큰 → {count(trimmed)}토큰, 남은 쌍 {(len(trimmed) - 2) // 2}개 (예측 {my_guess})')

## 실습 3-2. 누적 입력 토큰 계산

교재의 절: "개념 확인 실습"

질문 200토큰, 답 300토큰인 대화를 20번 이어 갈 때, 20번의 요청에서 보낸 입력 토큰의 합을 `total_input_tokens()`로 계산합니다. 시스템 프롬프트는 없다고 가정합니다. 이력을 모두 보낼 때와 최근 3쌍만 보낼 때의 합을 먼저 `my_guess`에 어림해 적고, 실행 결과가 예측과 다르면 그 이유를 한 문장으로 설명하면 끝납니다.

In [ ]:
def total_input_tokens(turns, question_tokens=200, answer_tokens=300, keep_last=None):
    """요청마다 보낸 입력 토큰(이력 + 새 질문)을 모두 더합니다. keep_last가 있으면 이력을 최근 keep_last쌍만 보냅니다"""
    return 0  # TODO


my_guess = {'전체 이력': None, '최근 3쌍만': None}  # TODO: 20번의 요청 동안 보낸 입력 토큰의 합을 어림해 적습니다
print(f'전체 이력: {total_input_tokens(20):,}토큰 (예측 {my_guess["전체 이력"]})')
print(f'최근 3쌍만: {total_input_tokens(20, keep_last=3):,}토큰 (예측 {my_guess["최근 3쌍만"]})')